In [ ]:
#| default_exp prompts

# prompts

> Prompt template loading and variable substitution.
>
> Templates live in `manhualizer/templates/<name>/`. Each template set is a directory of YAML files.
> Variable substitution uses simple `{variable}` syntax — no Jinja2 dependency.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
from pathlib import Path
from typing import Any
import yaml

In [ ]:
#| export
# Built-in templates ship inside the package.
# In notebook context __file__ is not defined, so we fall back to
# resolving relative to the installed package location.
try:
    _BUILTIN_TEMPLATES_DIR = Path(__file__).parent / "templates"
except NameError:
    import manhualizer as _pkg
    _BUILTIN_TEMPLATES_DIR = Path(_pkg.__file__).parent / "templates"

In [ ]:
#| export
class TemplateSet:
    """A loaded set of prompt templates from one directory.
    
    Template files are YAML. String values support `{variable}` substitution
    via `render(key, **vars)`. Missing variables leave the placeholder intact.

    If `fallback_dir` is given, any file not found in `templates_dir` is
    loaded from there instead. This lets style-only templates (e.g. cinematic,
    noir) override only `image_style.yml` while inheriting all other prompt
    files from the default template.
    """

    def __init__(self, templates_dir: Path, fallback_dir: Path | None = None):
        self._dir = templates_dir
        self._fallback_dir = fallback_dir
        self._cache: dict[str, dict] = {}

    def _load(self, filename: str) -> dict:
        if filename not in self._cache:
            path = self._dir / filename
            if not path.exists() and self._fallback_dir is not None:
                path = self._fallback_dir / filename
            if not path.exists():
                raise FileNotFoundError(f"Template not found: {path}")
            with open(path) as f:
                self._cache[filename] = yaml.safe_load(f) or {}
        return self._cache[filename]

    def get(self, filename: str, key: str) -> str:
        """Return the raw template string for `key` in `filename`."""
        data = self._load(filename)
        if key not in data:
            raise KeyError(f"Key '{key}' not found in template '{filename}'")
        return data[key]

    def render(self, filename: str, key: str, **variables: Any) -> str:
        """Return the template string with `{variable}` placeholders substituted."""
        template = self.get(filename, key)
        return _safe_substitute(template, variables)

    # Convenience accessors for the main template files
    @property
    def image_style(self) -> dict:
        return self._load("image_style.yml")

    @property
    def style_prefix(self) -> str:
        return self.image_style.get("style_prefix", "")

    @property
    def negative_prompt(self) -> str:
        return self.image_style.get("negative_prompt", "")

In [ ]:
#| export
import re

def _safe_substitute(template: str, variables: dict) -> str:
    """Substitute only simple `{identifier}` placeholders; leave all else intact.
    
    Unlike str.format_map(), this never recurses into complex `{...}` blocks
    (e.g. literal JSON examples in prompt templates).
    """
    def _replace(m: re.Match) -> str:
        key = m.group(1)
        return str(variables[key]) if key in variables else m.group(0)
    return re.sub(r'\{(\w+)\}', _replace, template)

In [ ]:
#| export
def load_templates(
    template_name: str = "default",
    custom_dir: str | Path | None = None,
) -> TemplateSet:
    """Load a TemplateSet by name.

    Resolution order:
    1. `custom_dir/<template_name>/` if provided
    2. Built-in `manhualizer/templates/<template_name>/`

    For style-only templates (e.g. cinematic, noir) that only override
    `image_style.yml`, all other prompt files fall back to the `default`
    built-in template automatically.
    """
    default_dir = _BUILTIN_TEMPLATES_DIR / "default"

    if custom_dir is not None:
        candidate = Path(custom_dir) / template_name
        if candidate.is_dir():
            return TemplateSet(candidate, fallback_dir=default_dir)

    builtin = _BUILTIN_TEMPLATES_DIR / template_name
    if not builtin.is_dir():
        raise ValueError(
            f"Unknown template '{template_name}'. "
            f"Built-in options: {list_builtin_templates()}"
        )
    # Default template has no fallback; non-default templates fall back to default.
    fallback = None if template_name == "default" else default_dir
    return TemplateSet(builtin, fallback_dir=fallback)


def list_builtin_templates() -> list[str]:
    """Return the names of all built-in template sets."""
    return sorted(p.name for p in _BUILTIN_TEMPLATES_DIR.iterdir() if p.is_dir())

In [ ]:
#| export
def build_panel_prompt(
    templates: TemplateSet,
    *,
    location_prompt: str,
    character_prompts: str,
    action: str,
    mood: str,
) -> str:
    """Assemble a final image-gen prompt for a single panel."""
    template = templates.image_style.get("panel_template", "{style_prefix}, {location_prompt}, {character_prompts}, {action}, {mood}")
    return template.format(
        style_prefix=templates.style_prefix,
        location_prompt=location_prompt,
        character_prompts=character_prompts,
        action=action,
        mood=mood,
    )


def build_character_sheet_prompt(
    templates: TemplateSet,
    *,
    character_description: str,
) -> str:
    """Assemble an image-gen prompt for a character reference sheet."""
    template = templates.image_style.get(
        "character_sheet_template",
        "{style_prefix}, character reference sheet, full body, {character_description}",
    )
    return template.format(
        style_prefix=templates.style_prefix,
        character_description=character_description,
    )

## Tests

In [ ]:
templates = load_templates("default")
assert "manhua" in templates.style_prefix
assert templates.negative_prompt

# render substitution
prompt = templates.render("analyze.yml", "analyze_prompt",
    story_chunk="Once upon a time...", prior_analysis="{}")
assert "Once upon a time" in prompt

# panel prompt assembly
panel_prompt = build_panel_prompt(
    templates,
    location_prompt="ancient temple, stone walls",
    character_prompts="young man with black hair",
    action="running through doorway",
    mood="tense",
)
assert "manhua" in panel_prompt
assert "ancient temple" in panel_prompt

# cinematic template
cin = load_templates("cinematic")
assert "cinematic" in cin.style_prefix

print("Templates OK")
print("Built-in templates:", list_builtin_templates())

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()